# 00 -- Journal pipeline demo (paired-script convention)

**Not one of the ten required notebooks from `TASK-028_PYTHON_STATISTICAL_LAB.md`.**
This is a minimal, deliberately thin demonstration of the reproducibility contract's
rule 1 ("every notebook has a paired `.py` pipeline containing the actual logic") using
the one script actually built so far, `analysis/join_trade_journal.py` -- it exists to
prove out the paired-notebook pattern end to end before the ten real notebooks are built
against it.

**Uses clearly-labelled SYNTHETIC fixture data only.** No real trade journal exists yet
from this project's own EA (runtime verification of `ThembaAdaptiveIntradayEA.mq5` is
still batched/pending per TASK-025 through TASK-027's own task files) -- per
reproducibility rule 7, this notebook marks the real-data run as pending rather than
fabricating one.

In [1]:
import json
import sys
import tempfile
from pathlib import Path

# This notebook lives in .../03_SOURCE_CODE/Python/notebooks/ -- add the
# Python project root (one level up) to sys.path so `analysis`/
# `data_collection` import exactly as they do for the paired .py pipeline
# and the pytest suite (see ../pyproject.toml's pythonpath setting, which
# only applies to pytest, not a notebook kernel -- hence doing it explicitly
# here instead of relying on hidden kernel state, per reproducibility rule 2).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.join_trade_journal import run

## Build a synthetic journal directory

Three records: one schema-valid, one deliberately schema-invalid (blank
`signal_id`/`market_family`/`intraday_mode` -- **corrected, 2026-07-27, Codex
round-8 P1 finding 20 fallout: this record was previously described as "shaped
exactly like what the CURRENT live EA build actually emits today," a claim
that stopped being true once TASK-040 wired `IntradayModeRouter.mqh` into the
journal (`market_family`/`intraday_mode` are populated on every real decision)
and TASK-036 wired `BuildSignalId` (`signal_id` likewise never blank) -- see
`analysis/schema.py`'s own docstring, which was corrected the same way in an
earlier round. It is kept as a purely synthetic, deliberately-invalid record
to exercise the validation-error reporting path below, not as a claim about
current EA behavior.**), and one malformed JSON line -- so this demo
exercises all three of the pipeline's own reporting paths (valid /
schema-invalid / parse-invalid) in one pass, not just the happy path.

In [2]:
def make_valid_record(**overrides):
    record = {
        "signal_id": "demo-1",
        "timestamp_utc": "2026-07-21T14:05:30Z",
        "symbol": "XAUUSD",
        "market_family": "METAL",
        "intraday_mode": "SCALP",
        "regime": "REGIME_TRENDING_UP",
        "regime_confidence": 72.5,
        "direction": "BUY",
        "strategy": "TrendFollowingStrategy",
        "setup": "TrendlinePullback",
        "candlestick_pattern": "BullishEngulfing",
        "chart_pattern": None,
        "score": 68.0,
        "score_breakdown": {"base": 68.0},
        "entry": 2350.55,
        "stop": 2345.10,
        "targets": [2361.45],
        "risk_percent": 0.3,
        # **Fixed, 2026-07-27 (Codex round-8 P1 finding 20): this was
        # lowercase "clear", but analysis/schema.py's news_state is a
        # strict Literal requiring exactly "CLEAR"/"BLACKOUT"/"UNKNOWN" --
        # the lowercase value failed schema validation outright, so this
        # supposedly-valid fixture reported zero valid records.
        "news_state": "CLEAR",
        "session_state": "london",
        "reasons_passed": ["daily_weekly_loss_caps_clear"],
        "reasons_rejected": [],
        "ea_version": "1.01-task027-order-submission-optional",
        "git_commit": "ddcee10",
    }
    record.update(overrides)
    return record


# Deliberately schema-invalid (blank signal_id/market_family/intraday_mode) --
# see the markdown cell above for why this is no longer described as
# "what the live EA currently emits."
schema_invalid_record = make_valid_record(
    signal_id="", market_family="", intraday_mode="", timestamp_utc="2026-07-21T15:00:00Z"
)

tmp_dir = Path(tempfile.mkdtemp(prefix="themba_journal_demo_"))
journal_path = tmp_dir / "decisions_20260721.jsonl"
with journal_path.open("w", encoding="utf-8") as fh:
    fh.write(json.dumps(make_valid_record()) + "\n")
    fh.write(json.dumps(schema_invalid_record) + "\n")
    fh.write("{not valid json,,,\n")

print(f"Synthetic journal written to: {journal_path}")

Synthetic journal written to: C:\Users\THEMBA~1\AppData\Local\Temp\themba_journal_demo__us5tm8u\decisions_20260721.jsonl


## Run the paired pipeline

Calls `analysis.join_trade_journal.run` directly -- the exact same function the CLI
script and the pytest suite both call, so this cell's output is reproducible against the
same synthetic input regardless of whether it runs here, from the command line, or under
pytest.

In [3]:
out_dir = tmp_dir / "out"
result = run(
    input_dir=tmp_dir,
    output_csv=out_dir / "journal.csv",
    output_json=out_dir / "journal.json",
    errors_json=out_dir / "errors.json",
    symbol="XAUUSD",
    broker="Deriv (demo)",
    seed=42,
    repo_path=PROJECT_ROOT.parents[1],  # .../Themba_EA_Improvement_Lab repo root
)

print(f"valid_records            = {len(result.read_result.valid_records)}")
print(f"parse_errors             = {len(result.read_result.parse_errors)}")
print(f"validation_errors        = {len(result.read_result.validation_errors)}")
print(f"dataset_hash             = {result.metadata.dataset_hash}")
print(f"git_commit                = {result.metadata.git_commit}")

assert len(result.read_result.valid_records) == 1, "expected exactly the one schema-valid record"
assert len(result.read_result.parse_errors) == 1, "expected exactly the one malformed JSON line"
assert len(result.read_result.validation_errors) == 1, (
    "expected exactly the one current-EA-shaped rejection"
)

valid_records            = 1
parse_errors             = 1
validation_errors        = 1
dataset_hash             = 9a6982456a8c478fd053ef5c5355b1e43b340700c22cbd7d46e017a9c1f92277
git_commit                = ca2cbe14e8ef3459dcf93d404425ffd22af4b209


## Inspect the validation-error report

**Corrected, 2026-07-27 (Codex round-8 P1 finding 20 fallout):** this cell
previously claimed the invalid record above is "shaped exactly like what
`ThembaAdaptiveIntradayEA.mq5` (TASK-025/027) actually emits today," and that
its rejection was "a genuine cross-layer gap for a future MQL5-side task to
close." Neither is true anymore -- TASK-040 wired `market_family`/
`intraday_mode` population and TASK-036 wired `signal_id` population, both
before this correction (see `analysis/schema.py`'s own docstring). The record
below is now a purely synthetic, deliberately-invalid fixture demonstrating
this pipeline's validation-error reporting path, not a report of a live gap.

In [4]:
errors_payload = json.loads((out_dir / "errors.json").read_text(encoding="utf-8"))
print(json.dumps(errors_payload["validation_errors"][0], indent=2))

{
  "source_file": "decisions_20260721.jsonl",
  "line_number": 2,
  "raw_record": {
    "signal_id": "",
    "timestamp_utc": "2026-07-21T15:00:00Z",
    "symbol": "XAUUSD",
    "market_family": "",
    "intraday_mode": "",
    "regime": "REGIME_TRENDING_UP",
    "regime_confidence": 72.5,
    "direction": "BUY",
    "strategy": "TrendFollowingStrategy",
    "setup": "TrendlinePullback",
    "candlestick_pattern": "BullishEngulfing",
    "chart_pattern": null,
    "score": 68.0,
    "score_breakdown": {
      "base": 68.0
    },
    "entry": 2350.55,
    "stop": 2345.1,
    "targets": [
      2361.45
    ],
    "risk_percent": 0.3,
    "news_state": "CLEAR",
    "session_state": "london",
    "reasons_passed": [
      "daily_weekly_loss_caps_clear"
    ],
    "reasons_rejected": [],
    "ea_version": "1.01-task027-order-submission-optional",
    "git_commit": "ddcee10"
  },
  "error": "3 validation errors for TradeDecision\nsignal_id\n  String should have at least 1 character [type=st

## Real-data run: PENDING

No real `decisions_*.jsonl` file exists yet -- runtime verification of the live EA is
still on this project's batched manual-verification backlog. Once that backlog item is
performed and a real journal file exists, re-run this notebook's second cell pointed at
the real `MQL5\Files\ThembaEA\Journal\` directory instead of the synthetic one, per
reproducibility rule 7.